# 02 — Training

Addestramento della U-Net 2D per restaurare mel spectrogrammi degradati da EnCodec.

**Flusso:**
1. Carica le 2000 coppie `.pt` da Google Drive
2. Split 90/10 deterministico train/val
3. Training con AMP, early stopping e checkpoint su Drive
4. Sanity check visivo ogni 10 epoche
5. Plot curve di loss finale

## Cella 0 — Configurazione

Unico punto di verità per tutti gli iperparametri. Modificare qui prima di ri-eseguire qualunque altra cella.

In [ ]:
# ── Percorsi ──────────────────────────────────────────────────────────────────
DRIVE_ROOT  = "/content/drive/MyDrive/audio-restoration"
DATA_DIR    = f"{DRIVE_ROOT}/data/train"
CKPT_DIR    = f"{DRIVE_ROOT}/checkpoints"
PLOTS_DIR   = f"{DRIVE_ROOT}/plots"

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE  = 16
NUM_EPOCHS  = 50
LR          = 1e-3
VAL_SPLIT   = 0.1
PATIENCE    = 10
SAVE_EVERY  = 5

# ── Audio (confermati da 01_data_pipeline) ────────────────────────────────────
N_MELS      = 128
T_FRAMES    = 376

print("Configurazione caricata.")
print(f"  BATCH_SIZE={BATCH_SIZE}  EPOCHS={NUM_EPOCHS}  LR={LR}")
print(f"  VAL_SPLIT={VAL_SPLIT}  PATIENCE={PATIENCE}  SAVE_EVERY={SAVE_EVERY}")

## Cella 1 — Installazione e import

Installa torchaudio e importa tutte le dipendenze necessarie.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torchaudio"])

import os
import glob
import time
import random
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

print(f"PyTorch {torch.__version__}")
print(f"CUDA disponibile: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Cella 2 — Mount Google Drive e creazione cartelle

I checkpoint e i plot vengono salvati su Drive per persistere tra le sessioni Colab.

In [ ]:
import os, glob
from google.colab import drive

DRIVE_ROOT = "/content/drive/MyDrive/audio-restoration"
DATA_DIR   = f"{DRIVE_ROOT}/data/train"
CKPT_DIR   = f"{DRIVE_ROOT}/checkpoints"
PLOTS_DIR  = f"{DRIVE_ROOT}/plots"

drive.mount("/content/drive", force_remount=False)

for d in [CKPT_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

n_pt = len(glob.glob(os.path.join(DATA_DIR, "pair_*.pt")))
print(f"Drive montato.")
print(f"  Coppie disponibili: {n_pt}")
print(f"  CKPT_DIR  : {CKPT_DIR}")
print(f"  PLOTS_DIR : {PLOTS_DIR}")

## Cella 3 — Dataset e DataLoader

`MelDataset` carica le coppie `.pt` on-demand per non saturare la RAM.
Lo split 90/10 è deterministico (seed fisso): stesso identico split ad ogni resume.

In [ ]:
import os, glob, random, torch
from torch.utils.data import Dataset, DataLoader

DRIVE_ROOT  = "/content/drive/MyDrive/audio-restoration"
DATA_DIR    = f"{DRIVE_ROOT}/data/train"
BATCH_SIZE  = 16
VAL_SPLIT   = 0.1


class MelDataset(Dataset):
    """Carica coppie (degraded_mel, clean_mel) da file .pt su Drive."""

    def __init__(self, file_paths: List[str]):
        self.paths = file_paths

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        data = torch.load(self.paths[idx], weights_only=True)
        return data["degraded_mel"].float(), data["clean_mel"].float()


all_paths = sorted(glob.glob(os.path.join(DATA_DIR, "pair_*.pt")))
assert len(all_paths) > 0, f"Nessun file .pt in {DATA_DIR}"

rng = random.Random(42)
shuffled = all_paths[:]
rng.shuffle(shuffled)

n_val       = max(1, int(len(shuffled) * VAL_SPLIT))
n_train     = len(shuffled) - n_val
train_paths = shuffled[:n_train]
val_paths   = shuffled[n_train:]

train_ds = MelDataset(train_paths)
val_ds   = MelDataset(val_paths)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, persistent_workers=True, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, persistent_workers=True, pin_memory=True
)

print(f"Train: {n_train} campioni  |  Val: {n_val} campioni")
xb, yb = next(iter(train_loader))
print(f"Batch degraded : {xb.shape}")
print(f"Batch clean    : {yb.shape}")

## Cella 4 — Architettura UNet2D

U-Net con 4 livelli encoder/decoder + bottleneck.

- **GroupNorm** invece di BatchNorm: stabile con batch piccoli su GPU singola
- **Residual learning**: il modello predice il residuo da sottrarre all'input degradato — converge più velocemente perché input e target sono già simili (LSD baseline = 0.696)
- **Parametri target**: 10–15 milioni

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple


class ConvBlock(nn.Module):
    """Due strati Conv2d → GroupNorm → GELU."""

    def __init__(self, in_ch: int, out_ch: int, groups: int = 8):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(min(groups, out_ch), out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(min(groups, out_ch), out_ch),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class DownBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.conv = ConvBlock(in_ch, out_ch)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        skip = self.conv(x)
        return self.pool(skip), skip


class UpBlock(nn.Module):
    def __init__(self, in_ch: int, skip_ch: int, out_ch: int):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.conv = ConvBlock(in_ch // 2 + skip_ch, out_ch)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))


class UNet2D(nn.Module):
    """
    U-Net 2D per restauro mel spectrogram.
    Input/Output: [B, 1, 128, 376]
    Canali: 1→32→64→128→256 | bottleneck 512 | decoder simmetrico
    Residual learning: output = input - unet(input)
    """

    def __init__(self):
        super().__init__()
        self.down1      = DownBlock(1,   32)
        self.down2      = DownBlock(32,  64)
        self.down3      = DownBlock(64,  128)
        self.down4      = DownBlock(128, 256)
        self.bottleneck = ConvBlock(256, 512)
        self.up4        = UpBlock(512, 256, 256)
        self.up3        = UpBlock(256, 128, 128)
        self.up2        = UpBlock(128, 64,  64)
        self.up1        = UpBlock(64,  32,  32)
        self.head       = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_in = x
        x, s1 = self.down1(x)
        x, s2 = self.down2(x)
        x, s3 = self.down3(x)
        x, s4 = self.down4(x)
        x = self.bottleneck(x)
        x = self.up4(x, s4)
        x = self.up3(x, s3)
        x = self.up2(x, s2)
        x = self.up1(x, s1)
        return x_in - self.head(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = UNet2D().to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parametri totali: {n_params:,}  ({n_params/1e6:.2f}M)")
assert 10e6 <= n_params <= 15e6, f"Parametri fuori range: {n_params/1e6:.2f}M"

dummy = torch.randn(2, 1, 128, 376, device=device)
with torch.no_grad():
    out = model(dummy)
assert out.shape == dummy.shape, f"Shape mismatch: {out.shape} != {dummy.shape}"
print(f"Input  : {dummy.shape}")
print(f"Output : {out.shape}")
print("Shape OK.")

## Cella 5 — Loss e ottimizzatore

- **L1Loss**: robusta agli outlier spettrali, adatta per artefatti distribuiti su tutto lo spettro (confermato da 01_data_pipeline, LSD baseline = 0.696)
- **AdamW** con weight_decay per regularizzazione leggera
- **ReduceLROnPlateau**: dimezza lr se val_loss non migliora per 5 epoche

In [ ]:
import torch.nn as nn

LR = 1e-3

criterion = nn.L1Loss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    patience=5,
    factor=0.5,
    verbose=True
)

print(f"Loss     : {criterion.__class__.__name__}")
print(f"Optimizer: AdamW  lr={LR}  weight_decay=1e-4")
print(f"Scheduler: ReduceLROnPlateau  patience=5  factor=0.5")

## Cella 6 — Resume da checkpoint

Carica automaticamente l'ultimo checkpoint da `CKPT_DIR` se esiste.
Permette di riprendere il training tra sessioni Colab senza perdere progresso.

In [ ]:
import os, glob, torch
from typing import Dict, Tuple

DRIVE_ROOT = "/content/drive/MyDrive/audio-restoration"
CKPT_DIR   = f"{DRIVE_ROOT}/checkpoints"


def load_checkpoint(
    ckpt_dir: str,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler,
    device: torch.device,
) -> Tuple[int, Dict]:
    """
    Carica l'ultimo checkpoint disponibile.
    Returns: (start_epoch, history)
    """
    ckpts = sorted(glob.glob(os.path.join(ckpt_dir, "ckpt_epoch_*.pth")))

    if not ckpts:
        print("Nessun checkpoint trovato — training da zero (epoca 1).")
        return 0, {"train_loss": [], "val_loss": []}

    last = ckpts[-1]
    print(f"Caricamento: {os.path.basename(last)}")
    ckpt = torch.load(last, map_location=device)

    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])

    start_epoch = ckpt["epoch"]
    history     = ckpt.get("history", {"train_loss": [], "val_loss": []})

    print(f"  Ultima epoca completata : {start_epoch}")
    if history["val_loss"]:
        print(f"  Best val_loss in storia : {min(history['val_loss']):.6f}")
    print(f"  Riprendo da epoca       : {start_epoch + 1}")
    return start_epoch, history


start_epoch, history = load_checkpoint(CKPT_DIR, model, optimizer, scheduler, device)

## Cella 8 — Sanity check visivo

Plotta **Degraded | Target Clean | Predicted** per un campione del validation set.
Verifica visivamente la qualità del restauro durante il training. Va eseguita **prima** della Cella 7.

In [ ]:
import os, torch, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from torch.cuda.amp import autocast

DRIVE_ROOT = "/content/drive/MyDrive/audio-restoration"
PLOTS_DIR  = f"{DRIVE_ROOT}/plots"


def plot_sanity_check(model: nn.Module, val_loader: DataLoader, epoch: int) -> None:
    """Salva figura 1x3: Degraded | Target Clean | Predicted."""
    model.eval()
    with torch.no_grad():
        degraded_b, clean_b = next(iter(val_loader))
        degraded_b = degraded_b.to(device)
        with autocast():
            pred_b = model(degraded_b)

    deg   = degraded_b[0, 0].cpu().numpy()
    clean = clean_b[0, 0].numpy()
    pred  = pred_b[0, 0].cpu().float().numpy()

    vmin = min(deg.min(), clean.min(), pred.min())
    vmax = max(deg.max(), clean.max(), pred.max())

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for ax, title, panel in zip(
        axes,
        ["Degraded (input)", "Target Clean", f"Predicted (epoch {epoch})"],
        [deg, clean, pred]
    ):
        im = ax.imshow(panel, aspect="auto", origin="lower",
                       vmin=vmin, vmax=vmax, cmap="magma")
        ax.set_title(title, fontsize=11)
        ax.set_xlabel("Frame")
        ax.set_ylabel("Banda Mel")
        plt.colorbar(im, ax=ax)

    l1_deg  = float(np.abs(deg  - clean).mean())
    l1_pred = float(np.abs(pred - clean).mean())
    fig.suptitle(
        f"Sanity check — Epoch {epoch}  |  "
        f"L1(deg,clean)={l1_deg:.4f}  L1(pred,clean)={l1_pred:.4f}",
        fontsize=12, fontweight="bold"
    )
    plt.tight_layout()

    save_path = os.path.join(PLOTS_DIR, f"sanity_epoch_{epoch:03d}.png")
    plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.close(fig)
    print(f"  Sanity check salvato: sanity_epoch_{epoch:03d}.png")
    model.train()


print("plot_sanity_check definita. Eseguire questa cella PRIMA della Cella 7.")

## Cella 7 — Training loop

- **AMP** (autocast + GradScaler): ~1.5× speedup e meno VRAM su T4
- **Gradient clipping** (max_norm=1.0): evita esplosione dei gradienti
- **Early stopping** dopo `PATIENCE` epoche senza miglioramento su val_loss
- **Checkpoint** ogni `SAVE_EVERY` epoche + best model separato come `best_unet.pth`
- Mantiene solo gli ultimi 3 checkpoint per non esaurire lo spazio Drive
- **Sanity check** ogni 10 epoche (richiede Cella 8 eseguita prima)

In [ ]:
import os, glob, time, torch
from torch.cuda.amp import autocast, GradScaler

DRIVE_ROOT  = "/content/drive/MyDrive/audio-restoration"
CKPT_DIR    = f"{DRIVE_ROOT}/checkpoints"
PLOTS_DIR   = f"{DRIVE_ROOT}/plots"
NUM_EPOCHS  = 50
PATIENCE    = 10
SAVE_EVERY  = 5
MAX_CKPTS   = 3

scaler     = GradScaler()
best_val   = min(history["val_loss"]) if history["val_loss"] else float("inf")
no_improve = 0
t0_total   = time.time()


def save_checkpoint(epoch: int, val_loss: float) -> str:
    path = os.path.join(CKPT_DIR, f"ckpt_epoch_{epoch:03d}.pth")
    torch.save({
        "epoch":           epoch,
        "model_state":     model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "val_loss":        val_loss,
        "history":         history,
    }, path)
    all_ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "ckpt_epoch_*.pth")))
    for old in all_ckpts[:-MAX_CKPTS]:
        os.remove(old)
    return path


def run_epoch(loader, train: bool) -> float:
    model.train(train)
    total = 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for degraded, clean in loader:
            degraded = degraded.to(device, non_blocking=True)
            clean    = clean.to(device, non_blocking=True)
            with autocast():
                pred = model(degraded)
                loss = criterion(pred, clean)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            total += loss.item()
    return total / len(loader)


print(f"Training da epoca {start_epoch + 1} / {NUM_EPOCHS}")
print(f"Best val_loss corrente: {best_val:.6f}")
print("-" * 70)

for epoch in range(start_epoch + 1, NUM_EPOCHS + 1):
    t0 = time.time()

    train_loss = run_epoch(train_loader, train=True)
    val_loss   = run_epoch(val_loader,   train=False)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch:3d}/{NUM_EPOCHS}  "
        f"train={train_loss:.6f}  val={val_loss:.6f}  "
        f"lr={current_lr:.2e}  [{time.time()-t0:.0f}s]"
    )

    if val_loss < best_val:
        best_val   = val_loss
        no_improve = 0
        torch.save(model.state_dict(), os.path.join(CKPT_DIR, "best_unet.pth"))
        print(f"  ✓ Nuovo best: {best_val:.6f} — best_unet.pth aggiornato")
    else:
        no_improve += 1

    if epoch % SAVE_EVERY == 0:
        ckpt_path = save_checkpoint(epoch, val_loss)
        print(f"  Checkpoint: {os.path.basename(ckpt_path)}")

    if epoch % 10 == 0:
        plot_sanity_check(model, val_loader, epoch)

    if no_improve >= PATIENCE:
        print(f"\nEarly stopping: nessun miglioramento per {PATIENCE} epoche.")
        break

print(f"\nTraining completato in {(time.time()-t0_total)/60:.1f} min.")
print(f"Best val_loss: {best_val:.6f}")
plot_sanity_check(model, val_loader, epoch)

## Cella 9 — Plot curve di loss

Visualizza train_loss e val_loss per epoca. Il gap tra le due curve indica overfitting.
Salvato su Drive come `loss_curve.png`.

In [ ]:
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DRIVE_ROOT = "/content/drive/MyDrive/audio-restoration"
PLOTS_DIR  = f"{DRIVE_ROOT}/plots"

assert history["train_loss"], "Nessuna storia trovata. Esegui prima la Cella 7."

epochs_r = range(1, len(history["train_loss"]) + 1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs_r, history["train_loss"], label="Train L1",      color="steelblue",  linewidth=2)
ax.plot(epochs_r, history["val_loss"],   label="Validation L1", color="darkorange", linewidth=2)

best_epoch = history["val_loss"].index(min(history["val_loss"])) + 1
best_v     = min(history["val_loss"])
ax.axvline(best_epoch, color="green", linestyle="--", alpha=0.7,
           label=f"Best epoch {best_epoch} (val={best_v:.4f})")

ax.set_xlabel("Epoca",   fontsize=12)
ax.set_ylabel("L1 Loss", fontsize=12)
ax.set_title("Curve di Loss — UNet2D Audio Restoration", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()

save_path = os.path.join(PLOTS_DIR, "loss_curve.png")
plt.savefig(save_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Salvato: {save_path}")
print(f"Best val_loss: {best_v:.6f} @ epoch {best_epoch}")